# LSTM中英翻译；

基于LSTM的英汉机器翻译项目，使用PyTorch实现 双向LSTM编码器和解码器，支持注意力机制。

In [152]:
import torch;
import torch.nn as nn;
import torch.optim as optim;
from torch.utils.data import Dataset, DataLoader;
import numpy as np;
import pandas as pd;
import jieba;
import re;
from collections import Counter;
import random;
import os;
from tqdm import tqdm;

# 设置随机种子。
SEED = 42;
torch.manual_seed(SEED);
np.random.seed(SEED);
random.seed(SEED);

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu');
print(f"使用设备: {device} !");

使用设备: cuda !


In [153]:
DATA_PATH = '/kaggle/input/datasets/ailananjuxi/zhongyingfanyi/data.xls';

# 词汇表参数。
MIN_FREQ = 2;           # 最小出现次数，用于过滤低频词。
MAX_VOCAB_SIZE = 50000; # # 最大词汇表大小。 

# 模型参数。
EMBED_DIM = 256;  # 嵌入维度。
HIDDEN_DIM = 512; # 隐藏层维度。
NUM_LAYERS = 2;   # LSTM层数。
DROPOUT = 0.3;

# 训练参数。
BATCH_SIZE = 64;
LEARNING_RATE = 0.001;
NUM_EPOCHS = 15;
TEACHER_FORCING_RATIO = 0.5;  # 教师强制比例。

# 特殊标记。
PAD_TOKEN = '<PAD>';
UNK_TOKEN = '<UNK>';
SOS_TOKEN = '<SOS>';
EOS_TOKEN = '<EOS>';

In [154]:
class Vocabulary: # 管理词汇到索引的映射。
    def __init__(self):
        self.word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1, SOS_TOKEN: 2, EOS_TOKEN: 3};
        self.idx2word = {0: PAD_TOKEN, 1: UNK_TOKEN, 2: SOS_TOKEN, 3: EOS_TOKEN};
        self.word_count = Counter();
        self.n_words = 4;
    
    def add_words(self, words):
        self.word_count.update(words);
    
    def build(self, min_freq=MIN_FREQ):
        for word, count in self.word_count.most_common(MAX_VOCAB_SIZE):
            if count >= min_freq:
                self.word2idx[word] = self.n_words;
                self.idx2word[self.n_words] = word;
                self.n_words += 1;
    
    def encode(self, words):
        indices = [self.word2idx.get(w, 1) for w in words] + [3];
        return indices;
    
    def decode(self, indices):
        words = [];
        for idx in indices:
            if idx == 3: break;
            if idx != 0: words.append(self.idx2word.get(idx, UNK_TOKEN));
        return words;

In [155]:
def prepare_data(file_path): # 加载数据并构建词汇表。

    print("正在加载数据...：");
    
    df = pd.read_csv(file_path, header=None);
    en_list, zh_list = df[0].astype(str).tolist(), df[1].astype(str).tolist();
    
    src_vocab, tgt_vocab = Vocabulary(), Vocabulary();
    pairs = [];
    
    print("正在处理句子...：");
    for en, zh in zip(en_list, zh_list):
        en_tokens = re.sub(r"[^a-z0-9\s]", " ", en.lower()).split();
        zh_tokens = [t for t in jieba.cut(zh) if t.strip()];
        
        if en_tokens and zh_tokens:
            pairs.append((en_tokens, zh_tokens));
            src_vocab.add_words(en_tokens);
            tgt_vocab.add_words(zh_tokens);
    
    src_vocab.build();
    tgt_vocab.build();
    
    print(f"英文词汇: {src_vocab.n_words}, 中文词汇: {tgt_vocab.n_words}, 句子对: {len(pairs)} !");
    return pairs, src_vocab, tgt_vocab;

In [156]:
class TranslationDataset(Dataset): #翻译数据集。

    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.data = [(src_vocab.encode(s), tgt_vocab.encode(t)) for s, t in pairs];
    
    def __len__(self):
        return len(self.data);
    
    def __getitem__(self, idx):
        src, tgt = self.data[idx];
        return torch.tensor(src), torch.tensor(tgt);


def collate_fn(batch): # 填充序列到相同长度。
    
    src_batch, tgt_batch = zip(*batch);
    src_len, tgt_len = [len(s) for s in src_batch], [len(t) for t in tgt_batch];
    
    src_pad = torch.zeros(len(batch), max(src_len), dtype=torch.long);
    for i, seq in enumerate(src_batch):
        src_pad[i, :len(seq)] = seq;
    
    tgt_pad = torch.zeros(len(batch), max(tgt_len), dtype=torch.long);
    for i, seq in enumerate(tgt_batch):
        tgt_pad[i, :len(seq)] = seq;
    
    return src_pad, tgt_pad, src_len, tgt_len;

### 加载数据:

In [157]:
pairs, src_vocab, tgt_vocab = prepare_data(DATA_PATH);

# 划分训练集和验证集。
split_idx = int(len(pairs) * 0.7);
train_pairs = pairs[:split_idx];
val_pairs = pairs[split_idx:];

print(f"训练集: {len(train_pairs)}, 验证集: {len(val_pairs)} !");

# 创建数据加载器。
train_dataset = TranslationDataset(train_pairs, src_vocab, tgt_vocab);
val_dataset = TranslationDataset(val_pairs, src_vocab, tgt_vocab);

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn);
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn);

# 显示几个样本。
print("\n样本数据:")
for i in range(3):
    en, zh = pairs[i];
    print(f"英文: {' '.join(en)}");
    print(f"中文: {''.join(zh)}");
    print();

正在加载数据...：
正在处理句子...：
英文词汇: 4425, 中文词汇: 6496, 句子对: 29155 !
训练集: 20408, 验证集: 8747 !

样本数据:
英文: hi
中文: 嗨。

英文: hi
中文: 你好。

英文: run
中文: 你用跑的。



## 模型:

In [158]:
class Encoder(nn.Module): # LSTM编码器。
    
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super(Encoder, self).__init__();
        self.hidden_dim = hidden_dim;
        self.num_layers = num_layers;
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0);
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True,
            batch_first=True
        );
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim);
        self.fc_cell = nn.Linear(hidden_dim * 2, hidden_dim);
        self.dropout = nn.Dropout(dropout);
    
    def forward(self, src, src_lengths):
        embedded = self.dropout(self.embedding(src));
        packed = nn.utils.rnn.pack_padded_sequence(embedded, src_lengths, batch_first=True, enforce_sorted=False);
        outputs, (hidden, cell) = self.lstm(packed);
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True);
        
        hidden = hidden.view(self.num_layers, 2, -1, self.hidden_dim);
        cell = cell.view(self.num_layers, 2, -1, self.hidden_dim);
        hidden = torch.cat((hidden[:, 0, :, :], hidden[:, 1, :, :]), dim=2);
        cell = torch.cat((cell[:, 0, :, :], cell[:, 1, :, :]), dim=2);
        
        hidden = self.fc_hidden(hidden);
        cell = self.fc_cell(cell);
        
        return outputs, hidden, cell;

In [159]:
class Attention(nn.Module): # 注意力机制。
    
    def __init__(self, hidden_dim):
        super(Attention, self).__init__();
        self.attn = nn.Linear(hidden_dim * 3, hidden_dim);
        self.v = nn.Linear(hidden_dim, 1, bias=False);
    
    def forward(self, hidden, encoder_outputs, mask=None):
        src_len = encoder_outputs.shape[1];
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1);
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)));
        attention = self.v(energy).squeeze(2);
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e10);
        return torch.softmax(attention, dim=1);

In [160]:
class Decoder(nn.Module): # LSTM解码器。
    
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, use_attention=True):
        super(Decoder, self).__init__();
        self.vocab_size = vocab_size;
        self.hidden_dim = hidden_dim;
        self.num_layers = num_layers;
        self.use_attention = use_attention;
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0);
        
        if use_attention:
            self.attention = Attention(hidden_dim);
            lstm_input_dim = embed_dim + hidden_dim * 2;
        else:
            lstm_input_dim = embed_dim;
        
        self.lstm = nn.LSTM(
            lstm_input_dim, hidden_dim, num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        );
        
        if use_attention:
            self.fc_out = nn.Linear(hidden_dim * 3, vocab_size);
        else:
            self.fc_out = nn.Linear(hidden_dim, vocab_size);
        
        self.dropout = nn.Dropout(dropout);
    
    def forward(self, input, hidden, cell, encoder_outputs=None, mask=None):
        input = input.unsqueeze(1);
        embedded = self.dropout(self.embedding(input));
        
        if self.use_attention and encoder_outputs is not None:
            attn_weights = self.attention(hidden[-1], encoder_outputs, mask);
            attn_weights = attn_weights.unsqueeze(1);
            context = torch.bmm(attn_weights, encoder_outputs);
            lstm_input = torch.cat((embedded, context), dim=2);
        else:
            lstm_input = embedded;
            context = None;
        
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell));
        
        if self.use_attention and context is not None:
            output = torch.cat((output.squeeze(1), context.squeeze(1)), dim=1);
        else:
            output = output.squeeze(1);
        
        prediction = self.fc_out(output);
        
        return prediction, hidden, cell;

In [161]:
class Seq2Seq(nn.Module): # 序列到序列模型。
    
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__();
        self.encoder = encoder;
        self.decoder = decoder;
        self.device = device;
    
    def forward(self, src, tgt, src_lengths, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0];
        tgt_len = tgt.shape[1];
        tgt_vocab_size = self.decoder.vocab_size;
        
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device);
        encoder_outputs, hidden, cell = self.encoder(src, src_lengths);
        
        input = torch.tensor([2] * batch_size).to(self.device);
        mask = (src != 0).to(self.device);
        
        for t in range(tgt_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs, mask);
            outputs[:, t, :] = output;
            teacher_force = random.random() < teacher_forcing_ratio;
            top1 = output.argmax(1);
            input = tgt[:, t] if teacher_force else top1;
        
        return outputs;
    
    def translate(self, src, src_vocab, tgt_vocab, src_lengths=None, max_len=50):
        self.eval();
        with torch.no_grad():
            if src_lengths is None:
                src_lengths = [len(src)];
            
            src = src.unsqueeze(0).to(self.device);
            encoder_outputs, hidden, cell = self.encoder(src, src_lengths);
            input = torch.tensor([2]).to(self.device);
            mask = (src != 0).to(self.device);
            
            outputs = [];
            for _ in range(max_len):
                output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs, mask);
                top1 = output.argmax(1).item();
                if top1 == 3:
                    break;
                outputs.append(top1);
                input = torch.tensor([top1]).to(self.device);
            
            return tgt_vocab.decode(outputs);

### 示例化模型:

In [162]:
encoder = Encoder(src_vocab.n_words, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT);
decoder = Decoder(tgt_vocab.n_words, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, use_attention=True);
model = Seq2Seq(encoder, decoder, device).to(device);

# 初始化权重。
def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name:
            nn.init.normal_(param.data, mean=0, std=0.01);
        else:
            nn.init.constant_(param.data, 0);

model.apply(init_weights);

# 计算参数量。
params = sum(p.numel() for p in model.parameters() if p.requires_grad);
print(f"模型参数量: {params} !");

模型参数量: 29846112 !


## 训练：

In [163]:
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE);
criterion = nn.CrossEntropyLoss(ignore_index=0);


def train_epoch(model, dataloader, optimizer, criterion, clip=1.0):
    model.train();
    epoch_loss = 0;
    
    pbar = tqdm(dataloader, desc="Training")
    for src, tgt, src_lengths, tgt_lengths in pbar:
        src = src.to(device);
        tgt = tgt.to(device);
        
        optimizer.zero_grad();
        output = model(src, tgt, src_lengths, TEACHER_FORCING_RATIO);
        
        output_dim = output.shape[-1];
        output = output[:, 1:].reshape(-1, output_dim);
        tgt = tgt[:, 1:].reshape(-1);
        
        loss = criterion(output, tgt);
        loss.backward();
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip);
        optimizer.step();
        
        epoch_loss += loss.item();
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return epoch_loss / len(dataloader);


def evaluate(model, dataloader, criterion):
    model.eval();
    epoch_loss = 0;
    
    with torch.no_grad():
        for src, tgt, src_lengths, tgt_lengths in dataloader:
            src = src.to(device);
            tgt = tgt.to(device);
            output = model(src, tgt, src_lengths, 0);
            
            output_dim = output.shape[-1];
            output = output[:, 1:].reshape(-1, output_dim);
            tgt = tgt[:, 1:].reshape(-1);
            
            loss = criterion(output, tgt);
            epoch_loss += loss.item();
    
    return epoch_loss / len(dataloader);

In [164]:
best_val_loss = float('inf');

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}");
    print("——" * 50);
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion);
    val_loss = evaluate(model, val_loader, criterion);
    
    print(f"训练损失: {train_loss:.4f} !");
    print(f"验证损失: {val_loss:.4f} !");
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss;
        torch.save({
            'model_state_dict': model.state_dict(),
            'src_vocab': src_vocab,
            'tgt_vocab': tgt_vocab,
        }, 'best_model.pth');
        print("✅保存最佳模型！");

print("\n✅训练完成！")


Epoch 1/15
————————————————————————————————————————————————————————————————————————————————————————————————————


Training: 100%|██████████| 319/319 [00:25<00:00, 12.53it/s, loss=4.3502]


训练损失: 4.8642 !
验证损失: 5.7000 !
✅保存最佳模型！

Epoch 2/15
————————————————————————————————————————————————————————————————————————————————————————————————————


Training: 100%|██████████| 319/319 [00:25<00:00, 12.63it/s, loss=3.7855]


训练损失: 3.9991 !
验证损失: 5.3063 !
✅保存最佳模型！

Epoch 3/15
————————————————————————————————————————————————————————————————————————————————————————————————————


...

训练损失: 2.1817 !
验证损失: 5.0742 !
✅保存最佳模型！

Epoch 6/15
————————————————————————————————————————————————————————————————————————————————————————————————————


Training: 100%|██████████| 319/319 [00:25<00:00, 12.67it/s, loss=2.2002]


训练损失: 1.7938 !
验证损失: 5.2847 !

Epoch 7/15
————————————————————————————————————————————————————————————————————————————————————————————————————


Training: 100%|██████████| 319/319 [00:25<00:00, 12.68it/s, loss=1.6558]


KeyboardInterrupt: 

## 测试：

In [165]:
def translate_sentence(model, sentence, src_vocab, tgt_vocab):
    tokens = re.sub(r"[^a-z0-9\s]", " ", sentence.lower()).split();
    indices = src_vocab.encode(tokens);
    src_tensor = torch.tensor(indices);
    translation = model.translate(src_tensor, src_vocab, tgt_vocab);
    return ''.join(translation);


test_sentences = [
    "Hello world",
    "How are you",
    "I love machine learning",
    "Thank you very much",
    "Good morning",
    "Nice to meet you",
];

print("\n" + "——" * 50);
print("翻译测试：");
print("——" * 50);

for sentence in test_sentences:
    translation = translate_sentence(model, sentence, src_vocab, tgt_vocab);
    print(f"英文: {sentence}");
    print(f"中文: {translation}");
    print("——" * 50);


————————————————————————————————————————————————————————————————————————————————————————————————————
翻译测试：
————————————————————————————————————————————————————————————————————————————————————————————————————
英文: Hello world
中文: 谢谢好。
————————————————————————————————————————————————————————————————————————————————————————————————————
英文: How are you
中文: 吗吗？
————————————————————————————————————————————————————————————————————————————————————————————————————
英文: I love machine learning
中文: 我爱冬天。
————————————————————————————————————————————————————————————————————————————————————————————————————
英文: Thank you very much
中文: 不是你简单。
————————————————————————————————————————————————————————————————————————————————————————————————————
英文: Good morning
中文: 会好。
————————————————————————————————————————————————————————————————————————————————————————————————————
英文: Nice to meet you
中文: 你你真你。
————————————————————————————————————————————————————————————————————————————————————————————————————


## 保存模型：

In [166]:
# 保存最终模型到Kaggle输出目录。
torch.save({
    'model_state_dict': model.state_dict(),
    'src_vocab': src_vocab,
    'tgt_vocab': tgt_vocab,
    'config': {
        'embed_dim': EMBED_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_layers': NUM_LAYERS,
        'dropout': DROPOUT,
    }
}, '/kaggle/working/final_model.pth');

print("✅模型已保存到 /kaggle/working/final_model.pth ！");

✅模型已保存到 /kaggle/working/final_model.pth ！
